# Debug slot probabilities (InterpretModel / policy selection)

Runs **one** forward on a checkpoint + snapshot: logs per-slot selection logits, **P(select)**, **argmax vs sample** behavior, and hand representation diversity. Same logic as ``env/debug_slot_probs.py``.

**Colab:** mount Drive in the next cell; set ``REPO_DIR`` to your clone (or ``BALATRO_AGENT_REPO``). PyTorch is preinstalled on Colab.

**Selection:** ``STOCHASTIC = True`` (default) uses **``Categorical.sample()``** per slot, same as ``TrainCombat`` / PPO. Set ``False`` only if you want deterministic **argmax** (often yields no selections when P(select)&lt;0.5 everywhere).

**Logs:** NDJSON appended to ``<REPO_ROOT>/.cursor/debug-992ac8.log`` (session id ``992ac8``). Download or ``cat`` that file after the run cell.


In [6]:
import os
import sys

try:
    from google.colab import drive

    drive.mount("/content/drive")
except ImportError:
    pass  # local Jupyter: set REPO_DIR

REPO_DIR = os.environ.get(
    "BALATRO_AGENT_REPO",
    "/content/drive/MyDrive/balatro-agent-project",
)
REPO_ROOT = REPO_DIR

os.chdir(REPO_DIR)

for p in (
    REPO_DIR,
    os.path.join(REPO_DIR, "balatro_lite_gym"),
):
    if p not in sys.path:
        sys.path.insert(0, p)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import json
import time
from pathlib import Path

import numpy as np
import torch
from torch.distributions import Categorical

from agent.model import CombatPPOAgent
from agent.ppo import get_card_mask, mask_logits
from agent.ppo_config import PPOConfig
from env.game_simulator import GameSimulator
from env.lite_combat_env import adapt_lite_vector_obs, dict_to_tensors


In [8]:
# --- edit ---
CKPT_PATH = Path(REPO_ROOT) / "checkpoints" / "combat_ppo_iter_400.pt"
SNAPSHOT_PATH = Path(REPO_ROOT) / "snapshots" / "snapshot_no_jokers_level1.json"
STOCHASTIC = True  # True = sample from softmax (default, matches training); False = argmax per slot
DEBUG_SESSION_ID = "992ac8"

_DEBUG_LOG = Path(REPO_ROOT) / ".cursor" / f"debug-{DEBUG_SESSION_ID}.log"

ckpt_path = CKPT_PATH.resolve()
snap_path = SNAPSHOT_PATH.resolve()
assert ckpt_path.is_file(), ckpt_path
assert snap_path.is_file(), snap_path


In [9]:
def _nested_obs_add_batch_dim(obs_raw: dict) -> dict:
    out: dict = {}
    for k, v in obs_raw.items():
        if isinstance(v, dict):
            out[k] = {
                kk: np.asarray(vv, dtype=np.float32)[np.newaxis, ...]
                for kk, vv in v.items()
            }
        else:
            out[k] = np.asarray(v, dtype=np.float32)[np.newaxis, ...]
    return out


def _dbg(hypothesis_id: str, message: str, data: dict) -> None:
    payload = {
        "sessionId": DEBUG_SESSION_ID,
        "hypothesisId": hypothesis_id,
        "timestamp": int(time.time() * 1000),
        "message": message,
        "data": data,
    }
    _DEBUG_LOG.parent.mkdir(parents=True, exist_ok=True)
    with open(_DEBUG_LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps(payload, default=str) + "\n")


In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_dbg("H2", "run_config", {"stochastic": STOCHASTIC, "device": str(device)})

try:
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
except TypeError:
    ckpt = torch.load(ckpt_path, map_location=device)
cfg = ckpt.get("config") or PPOConfig()
agent = CombatPPOAgent(
    d_model=cfg.d_model,
    nhead=cfg.nhead,
    dim_ff=cfg.dim_ff,
    dropout=cfg.dropout,
).to(device)
agent.load_state_dict(ckpt["model_state_dict"])
agent.eval()

sim = GameSimulator.from_json(snap_path, seed=0)
obs_t = dict_to_tensors(
    adapt_lite_vector_obs(_nested_obs_add_batch_dim(sim.obs)),
    device,
)

with torch.no_grad():
    sel_logits, exec_logits, value = agent(obs_t)
card_mask = get_card_mask(obs_t)
masked = mask_logits(sel_logits, card_mask)
slot_p = torch.softmax(masked, dim=-1)[0, :, 1].cpu().numpy()

n_real = int(card_mask[0].sum().item())
raw_slice = sel_logits[0, :n_real, :].cpu().numpy().tolist()
masked_slice = masked[0, :n_real, :].cpu().numpy().tolist()
p_slice = slot_p[:n_real].tolist()

_dbg("H1", "logits_per_slot_first_n", {"n_real": n_real, "raw": raw_slice, "masked": masked_slice})
_dbg("H3", "softmax_p_select_first_n", {"p": p_slice, "p_std": float(np.std(p_slice))})

if STOCHASTIC:
    sel_dist = Categorical(logits=masked)
    card_sels_s = sel_dist.sample()
    mode = "sample"
else:
    card_sels_s = masked.argmax(dim=-1)
    mode = "argmax"
n_sel = int(card_sels_s[0].sum().item())
sel_vec = card_sels_s[0].cpu().numpy().tolist()

_dbg(
    "H2",
    "discrete_selection",
    {"mode": mode, "n_sel": n_sel, "card_sels": sel_vec[: max(n_real, 8)]},
)

with torch.no_grad():
    pack = agent.embeddings(obs_t)
    hand_toks = pack[0]
    hf_in = hand_toks[0, :n_real, :]
    div_in = float(hf_in.std(dim=0).mean().item())
    hand_final, _ = agent.backbone(*pack)
    hf_out = hand_final[0, :n_real, :]
    div_out = float(hf_out.std(dim=0).mean().item())
_dbg(
    "H4",
    "hand_repr_diversity",
    {"n_real": n_real, "embed_mean_std_dim": div_in, "hand_final_mean_std_dim": div_out},
)

print("log file:", _DEBUG_LOG)
print("mode:", mode, "| n_sel:", n_sel, "| p_std:", float(np.std(p_slice)))
print("P(select) first n_real slots:", [round(x, 4) for x in p_slice])
print(
    "H4 mean std across hand slots (embed -> after backbone):",
    round(div_in, 4),
    "->",
    round(div_out, 4),
)
if not STOCHASTIC:
    print(
        "Note: argmax picks class 0 (no select) on a slot when P(select)<0.5;",
        "use STOCHASTIC=True for probability-based actions (PPO default).",
    )


log file: /content/drive/MyDrive/balatro-agent-project/.cursor/debug-992ac8.log
mode: sample | n_sel: 2 | p_std: 0.00016014381193038324
P(select) first n_real slots: [0.3814, 0.3811, 0.3811, 0.3809, 0.3813, 0.3813, 0.3815, 0.3811]
H4 mean std across hand slots (embed -> after backbone): 0.5935 -> 0.0166


Optional: open ``_DEBUG_LOG`` in the file browser or run ``!tail -n 12 "$DEBUG_LOG"`` after setting a shell variable to the printed path. The run cell above prints the important H4 summary.
